# PulmoSight OSIC CT Alternative Pretrained Feature Extractors (Kaggle GPU Notebook)

Extracts CT embeddings for the OSIC Pulmonary Fibrosis Progression dataset using alternative pretrained medical vision backbones so they can be compared directly against the baseline ImageNet ResNet-18 embeddings in `backend/src/imaging_progression.py`.

**Design principle:** Only the image feature extractor changes; DICOM loading (`load_dicom_as_hu`), lung segmentation (`segment_lungs_hu`), slice Z-sorting, and the canonical 175-patient cohort match `backend/src/imaging_progression.py` verbatim (without modifying project source files).

### Expected Kaggle Environment & Inputs
- **OSIC DICOM scans:** `/kaggle/input/competitions/osic-pulmonary-fibrosis-progression/train/<PatientID>/*.dcm`
- **PulmoSight backend dataset:** `/kaggle/input/datasets/<KAGGLE_USERNAME>/pulmosight-backend/backend` (copied to `/kaggle/working/project` and added to `sys.path`)
- **Accelerator & Network:** GPU enabled, Internet enabled
- **Hugging Face authentication:** Kaggle Secret named `HF_TOKEN` (read via `kaggle_secrets.UserSecretsClient`; never printed or saved)

> **Setup Note — Set Your Kaggle Username First:** Before running this notebook on Kaggle, edit `KAGGLE_USERNAME = "YOUR_KAGGLE_USERNAME"` at the top of **Cell 1** to your actual Kaggle username so `/kaggle/input/datasets/<KAGGLE_USERNAME>/pulmosight-backend/backend` resolves cleanly.

### Extractors Included (Isolated Sections)
Each extractor runs inside an independent guarded section so a failure in one extractor (for example, gated access or GPU memory limits) does not prevent the others from completing:

| Section | Extractor Key | Model / Source | Output CSV | Embedding Columns | Dim |
|---|---|---|---|---|---|
| 1 | `med3d` | MedicalNet (Med3D) 3D ResNet-18 (`TencentMedicalNet/MedicalNet-Resnet18`, `resnet_18_23dataset.pth`) | `cached_dicom_features_med3d.csv` | `med3d_emb_0..511` | 512 |
| 2 | `radimagenet` | RadImageNet 2D ResNet-50 (`Lab-Rasool/RadImageNet`, `ResNet50.pt`, strict key check) | `cached_dicom_features_radimagenet.csv` | `radimagenet_emb_0..2047` | 2048 |
| 3 | `ctclip` | CT-CLIP 3D CT-ViT image encoder (`ibrahimhamamci/CT-RATE` `CT-CLIP_v2.pt` + `ibrahimethemhamamci/CT-CLIP`) | `cached_dicom_features_ctclip.csv` | `ctclip_emb_0..511` | 512 |
| 4 (Optional Control) | `resnet50_imagenet_control` | **Extra Control:** ImageNet 2D ResNet-50 (`torchvision` `ResNet50_Weights.DEFAULT`) | `cached_dicom_features_resnet50_imagenet_control.csv` | `resnet50_imagenet_control_emb_0..2047` | 2048 |

Each extractor also writes a JSON metadata file (`/kaggle/working/cached_dicom_features_<extractor>_metadata.json`), performs an upfront one-patient timing test, checkpoints every 10 patients for crash-safe resume, and runs end-of-run NaN / constant-column / schema validation.

In [ ]:
# Cell 1: Verify Kaggle inputs, copy PulmoSight backend to /kaggle/working/project, add to sys.path, and load HF_TOKEN
import os
import shutil
import sys
from pathlib import Path

KAGGLE_USERNAME = "YOUR_KAGGLE_USERNAME"  # Set this to your Kaggle username before running
assert KAGGLE_USERNAME and KAGGLE_USERNAME != "YOUR_KAGGLE_USERNAME", (
    "Replace KAGGLE_USERNAME = 'YOUR_KAGGLE_USERNAME' in Cell 1 with your actual Kaggle username."
)

OSIC_TRAIN_DIR = Path("/kaggle/input/competitions/osic-pulmonary-fibrosis-progression/train")
BACKEND_INPUT_DIR = Path(f"/kaggle/input/datasets/{KAGGLE_USERNAME}/pulmosight-backend/backend")
WORK_DIR = Path("/kaggle/working")
WORK_PROJECT_DIR = WORK_DIR / "project"

assert OSIC_TRAIN_DIR.exists(), (
    f"OSIC DICOM directory not found at {OSIC_TRAIN_DIR}. "
    "Attach the 'osic-pulmonary-fibrosis-progression' competition dataset via Add Input."
)
assert BACKEND_INPUT_DIR.exists(), (
    f"PulmoSight backend dataset not found at {BACKEND_INPUT_DIR}. "
    f"Attach the '{KAGGLE_USERNAME}/pulmosight-backend' dataset via Add Input (and verify KAGGLE_USERNAME)."
)

# Copy read-only backend code into /kaggle/working/project so `backend.src.*` imports cleanly
if WORK_PROJECT_DIR.exists():
    shutil.rmtree(WORK_PROJECT_DIR)
WORK_PROJECT_DIR.mkdir(parents=True, exist_ok=True)

if (BACKEND_INPUT_DIR / "src" / "imaging_progression.py").exists():
    shutil.copytree(BACKEND_INPUT_DIR, WORK_PROJECT_DIR / "backend", dirs_exist_ok=True)
elif (BACKEND_INPUT_DIR / "backend" / "src" / "imaging_progression.py").exists():
    shutil.copytree(BACKEND_INPUT_DIR, WORK_PROJECT_DIR, dirs_exist_ok=True)
else:
    available = [p.name for p in sorted(BACKEND_INPUT_DIR.iterdir())]
    raise FileNotFoundError(
        f"Could not locate src/imaging_progression.py under {BACKEND_INPUT_DIR}. Found: {available}"
    )

backend_init = WORK_PROJECT_DIR / "backend" / "__init__.py"
if not backend_init.exists():
    backend_init.write_text("", encoding="utf-8")

if str(WORK_PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_PROJECT_DIR))

assert (WORK_PROJECT_DIR / "backend" / "src" / "imaging_progression.py").exists()
print(f"Verified OSIC scans directory : {OSIC_TRAIN_DIR}")
print(f"Copied PulmoSight backend to  : {WORK_PROJECT_DIR / 'backend'}")
print(f"Added to sys.path[0]          : {WORK_PROJECT_DIR}")

# Read Hugging Face token from Kaggle Secrets (never print or serialize the secret value)
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient

    _secret_val = UserSecretsClient().get_secret("HF_TOKEN")
    if _secret_val and str(_secret_val).strip():
        HF_TOKEN = str(_secret_val).strip()
        os.environ["HF_TOKEN"] = HF_TOKEN
        os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
        print("Hugging Face secret           : HF_TOKEN loaded from Kaggle Secrets (value hidden).")
    else:
        print("Hugging Face secret           : HF_TOKEN is empty (public models work; gated CT-CLIP will fail access).")
except Exception as exc:
    print(f"Hugging Face secret           : Could not load HF_TOKEN ({type(exc).__name__}); public models will still run.")

In [ ]:
# Cell 2: Install missing core dependencies (including DICOM pixel decoders and huggingface_hub) and verify GPU
import importlib.util
import random
import subprocess
import sys

CORE_PACKAGES = {
    "pydicom": "pydicom",
    "pylibjpeg": "pylibjpeg",
    "libjpeg": "pylibjpeg-libjpeg",
    "openjpeg": "pylibjpeg-openjpeg",
    "gdcm": "python-gdcm",
    "huggingface_hub": "huggingface_hub",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "xgboost": "xgboost",
    "tqdm": "tqdm",
    "torch": "torch",
    "torchvision": "torchvision",
}

missing_pkgs = [
    pip_pkg for mod_name, pip_pkg in CORE_PACKAGES.items()
    if importlib.util.find_spec(mod_name) is None
]
if missing_pkgs:
    print("Installing missing core packages:", missing_pkgs)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_pkgs])
else:
    print("All core packages are already installed.")

import numpy as np
import pandas as pd
import pydicom
import torch
import torchvision

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU-only"
GPU_TOTAL_MEM_GB = (
    round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2)
    if DEVICE.type == "cuda"
    else 0.0
)

print(f"PyTorch      : {torch.__version__} | torchvision: {torchvision.__version__} | pydicom: {pydicom.__version__}")
print(f"Active device: {DEVICE} ({GPU_NAME}, {GPU_TOTAL_MEM_GB} GB VRAM)")

In [ ]:
# Cell 3: Import unchanged PulmoSight functions and lock onto the exact 175 usable patients from cached_dicom_features.csv
from backend.src.imaging_progression import (
    load_dicom_as_hu,
    segment_lungs_hu,
)

EXISTING_CACHE_CSV = WORK_PROJECT_DIR / "backend" / "data" / "osic" / "cached_dicom_features.csv"
assert EXISTING_CACHE_CSV.exists(), (
    f"Existing baseline cache not found at {EXISTING_CACHE_CSV}. "
    "Ensure backend/data/osic/cached_dicom_features.csv is included in the backend dataset."
)

df_baseline_cache = pd.read_csv(EXISTING_CACHE_CSV, usecols=["Patient"])
TARGET_PATIENTS = sorted(df_baseline_cache["Patient"].astype(str).unique().tolist())

missing_scan_dirs = [pid for pid in TARGET_PATIENTS if not (OSIC_TRAIN_DIR / pid).is_dir()]
print(f"Canonical patients from existing cache : {len(TARGET_PATIENTS)}")
print(f"First 5 target patients                : {TARGET_PATIENTS[:5]}")
print(f"Last 5 target patients                 : {TARGET_PATIENTS[-5:]}")
print(f"Missing DICOM directories on disk      : {len(missing_scan_dirs)}")
if missing_scan_dirs:
    print("  Missing IDs:", missing_scan_dirs[:10])
assert len(TARGET_PATIENTS) == 175, f"Expected 175 usable patients from baseline cache, got {len(TARGET_PATIENTS)}"
assert len(missing_scan_dirs) == 0, f"Missing {len(missing_scan_dirs)} patient scan directories under {OSIC_TRAIN_DIR}"

In [ ]:
# Cell 4: Shared extraction, checkpoint/resume, timing-test, validation, and metadata JSON utilities
import gc
import hashlib
import json
import time
from collections import Counter
from datetime import datetime, timezone
from typing import Any, Callable, Dict, List, Optional, Tuple

import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm

CHECKPOINT_EVERY = 10
SLICE_BATCH_SIZE_2D = 64
EMBED_ROUND_DECIMALS = 6

# Exact ImageNet constants from backend/src/imaging_progression.py
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(3, 1, 1)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(3, 1, 1)

# Registry populated as each extractor section completes or records not_run/failed
EXTRACTOR_RUN_REGISTRY: Dict[str, Dict[str, Any]] = {}


def sanitize_message(msg: str) -> str:
    """Ensure no Hugging Face secret token ever appears in logs or metadata JSON."""
    out = str(msg)
    if HF_TOKEN and HF_TOKEN in out:
        out = out.replace(HF_TOKEN, "[REDACTED_HF_TOKEN]")
    return out


def compute_sha256(file_path: Path) -> str:
    """Compute hex SHA-256 checksum of a weights file for reproducibility."""
    h = hashlib.sha256()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def safe_torch_load(path: Path) -> Any:
    """Load a PyTorch checkpoint on CPU with weights_only=True first and safe dict fallback for legacy checkpoints."""
    try:
        return torch.load(str(path), map_location="cpu", weights_only=True)
    except TypeError:
        return torch.load(str(path), map_location="cpu")
    except Exception as exc:
        print(f"  [INFO] weights_only=True failed for {path.name} ({type(exc).__name__}); retrying legacy CPU load.")
        obj = torch.load(str(path), map_location="cpu", weights_only=False)
        if not isinstance(obj, dict):
            raise RuntimeError(
                f"Refusing to use non-dict checkpoint object ({type(obj).__name__}) from {path.name}."
            ) from exc
        return obj


def get_hf_commit_sha(repo_id: str, repo_type: str = "model", token: Optional[str] = None) -> Optional[str]:
    """Query Hugging Face Hub API for the resolved commit SHA of a repo."""
    try:
        from huggingface_hub import HfApi

        info = HfApi().repo_info(repo_id=repo_id, repo_type=repo_type, token=token)
        return getattr(info, "sha", None)
    except Exception:
        return None


def load_patient_sorted_slices(
    patient_id: str,
) -> Tuple[List[Tuple[np.ndarray, Tuple[float, float], float, float]], Optional[float], int]:
    """
    Load all readable DICOM slices for a patient using `backend.src.imaging_progression.load_dicom_as_hu`,
    sort along Z-position, and compute median inter-slice spacing `median_dz` identically to
    `extract_ct_features_from_3d_slices`.
    """
    pat_dir = OSIC_TRAIN_DIR / patient_id
    dcm_files = sorted(pat_dir.glob("*.dcm"))
    if not dcm_files:
        raise ValueError(f"No .dcm files found in {pat_dir}")

    slice_tuples = []
    slice_errors = 0
    last_err = ""
    for dcm_f in dcm_files:
        try:
            slice_tuples.append(load_dicom_as_hu(dcm_f))
        except Exception as exc:
            slice_errors += 1
            last_err = f"{type(exc).__name__}: {exc}"

    if not slice_tuples:
        raise RuntimeError(
            f"All {len(dcm_files)} DICOM slices failed to decode (last error: {last_err})"
        )

    sorted_slices = sorted(slice_tuples, key=lambda item: item[3])
    n_slices = len(sorted_slices)
    z_positions = [item[3] for item in sorted_slices]
    z_diffs = [
        abs(z_positions[i + 1] - z_positions[i])
        for i in range(n_slices - 1)
        if abs(z_positions[i + 1] - z_positions[i]) > 0.1
    ]
    median_dz = float(np.median(z_diffs)) if z_diffs else None
    return sorted_slices, median_dz, slice_errors


def preprocess_masked_slice_for_2d_cnn(masked_hu: np.ndarray) -> np.ndarray:
    """
    Bit-for-bit replica of `extract_pretrained_resnet18_slice_embedding` preprocessing in
    `backend/src/imaging_progression.py`:
      - Lung-masked HU slice (outside lung already set to -1000 HU)
      - Window -1000 to 0 HU -> [0, 1]
      - Convert to uint8 -> PIL resize to 224x224 (BILINEAR) -> / 255.0
      - Repeat grayscale to 3 channels and apply ImageNet mean/std normalization
    """
    norm_img = np.clip((masked_hu + 1000.0) / 1000.0, 0.0, 1.0).astype(np.float32)
    pil_img = Image.fromarray((norm_img * 255.0).astype(np.uint8)).resize(
        (224, 224), Image.Resampling.BILINEAR
    )
    arr_224 = np.asarray(pil_img, dtype=np.float32) / 255.0
    rgb = np.repeat(arr_224[np.newaxis, :, :], 3, axis=0)
    return (rgb - IMAGENET_MEAN) / IMAGENET_STD


def extract_2d_backbone_patient_embedding(
    sorted_slices: List[Tuple[np.ndarray, Tuple[float, float], float, float]],
    model: nn.Module,
    embed_dim: int,
    batch_size: int = SLICE_BATCH_SIZE_2D,
) -> np.ndarray:
    """
    Apply `segment_lungs_hu`, lung masking (-1000 HU outside lung), exact 224x224 ImageNet-normalized
    windowing, GPU-batched 2D CNN inference, and float64 mean pooling over all slices.
    """
    embed_sum = np.zeros(embed_dim, dtype=np.float64)
    embed_count = 0
    batch_buf: List[np.ndarray] = []

    def _flush() -> None:
        nonlocal embed_sum, embed_count
        if not batch_buf:
            return
        x = torch.from_numpy(np.stack(batch_buf, axis=0)).to(DEVICE, non_blocking=True)
        with torch.no_grad():
            out = model(x)
            if out.ndim > 2:
                out = torch.flatten(out, 1)
            embs = out.detach().cpu().numpy()
        embed_sum += embs.sum(axis=0, dtype=np.float64)
        embed_count += embs.shape[0]
        batch_buf.clear()

    for hu_img, _, _, _ in sorted_slices:
        mask = segment_lungs_hu(hu_img)
        masked_hu = np.where(mask, hu_img, -1000.0)
        batch_buf.append(preprocess_masked_slice_for_2d_cnn(masked_hu))
        if len(batch_buf) >= batch_size:
            _flush()
    _flush()

    if embed_count == 0:
        raise RuntimeError("Zero slice embeddings produced.")
    return (embed_sum / float(embed_count)).astype(np.float32)


def stack_slices_consistent_hw(slices_2d: List[np.ndarray], mode: str = "bilinear") -> np.ndarray:
    """Stack 2D axial slices into a (D, H, W) float32 volume, resizing any rare outlier slice to modal (H, W)."""
    shapes = [s.shape for s in slices_2d]
    modal_hw = Counter(shapes).most_common(1)[0][0]
    aligned = []
    for s in slices_2d:
        arr = np.asarray(s, dtype=np.float32)
        if arr.shape != modal_hw:
            t = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0)
            if mode == "nearest":
                t = F.interpolate(t, size=modal_hw, mode="nearest")
            else:
                t = F.interpolate(t, size=modal_hw, mode=mode, align_corners=False)
            arr = t[0, 0].numpy()
        aligned.append(arr)
    return np.stack(aligned, axis=0)


def save_embedding_checkpoint(
    completed_rows: Dict[str, Dict[str, Any]],
    expected_columns: List[str],
    out_csv: Path,
) -> None:
    """Atomically save sorted rows with exact ['Patient', '<extractor>_emb_*'] column order."""
    if not completed_rows:
        return
    ordered = [completed_rows[pid] for pid in sorted(completed_rows.keys())]
    df_out = pd.DataFrame(ordered)[expected_columns]
    tmp_path = out_csv.with_suffix(".tmp.csv")
    df_out.to_csv(tmp_path, index=False)
    tmp_path.replace(out_csv)


def validate_and_summarize_csv(
    output_csv: Path,
    extractor_key: str,
    embed_dim: int,
    skipped_patients: List[Dict[str, str]],
) -> Dict[str, Any]:
    """
    Validate output CSV schema and print required end-of-run checks:
    patient count, file size, first 3 rows, NaN/Inf check, and constant-column check.
    """
    emb_cols = [f"{extractor_key}_emb_{i}" for i in range(embed_dim)]
    expected_cols = ["Patient"] + emb_cols

    df = pd.read_csv(output_csv)
    file_size_mb = output_csv.stat().st_size / (1024.0 * 1024.0)

    if list(df.columns) != expected_cols:
        raise AssertionError(
            f"Column mismatch in {output_csv.name}: got {len(df.columns)} cols "
            f"(first 5: {list(df.columns[:5])}), expected {len(expected_cols)} cols."
        )

    emb_vals = df[emb_cols].to_numpy(dtype=np.float64)
    nan_cell_count = int(np.isnan(emb_vals).sum())
    inf_cell_count = int(np.isinf(emb_vals).sum())
    nan_cols = [c for c in emb_cols if df[c].isna().any()]
    all_nan_cols = [c for c in emb_cols if df[c].isna().all()]
    const_cols = [c for c in emb_cols if df[c].nunique(dropna=False) <= 1] if len(df) > 1 else []

    print("=" * 78)
    print(f"VALIDATION SUMMARY: {extractor_key}")
    print("=" * 78)
    print(f"Output CSV             : {output_csv}")
    print(f"File size              : {file_size_mb:.3f} MB")
    print(f"Patients saved         : {len(df)} / {len(TARGET_PATIENTS)}")
    print(f"Patients skipped       : {len(skipped_patients)}")
    if skipped_patients:
        for item in skipped_patients[:10]:
            print(f"  - {item['patient']}: {item['reason']}")
    print(f"Schema check           : PASS ({len(df.columns)} cols = Patient + {embed_dim} {extractor_key}_emb_*)")
    print(f"NaN check              : {nan_cell_count} NaN cells across {len(nan_cols)} cols (all-NaN cols: {len(all_nan_cols)})")
    print(f"Inf check              : {inf_cell_count} non-finite (+/-Inf) cells")
    print(f"Constant column check  : {len(const_cols)} constant embedding columns"
          + (f" (sample: {const_cols[:8]})" if const_cols else ""))

    preview_cols = ["Patient"] + emb_cols[:8]
    print(f"\nFirst 3 rows (Patient + first 8 of {embed_dim} embedding columns):")
    print(df[preview_cols].head(3).to_string(index=False))
    print("=" * 78)

    return {
        "file_size_mb": round(file_size_mb, 4),
        "row_count": int(len(df)),
        "column_count": int(len(df.columns)),
        "nan_cell_count": nan_cell_count,
        "nan_columns_count": len(nan_cols),
        "all_nan_columns_count": len(all_nan_cols),
        "inf_cell_count": inf_cell_count,
        "constant_columns_count": len(const_cols),
        "constant_columns_sample": const_cols[:10],
    }


def write_metadata_json(meta: Dict[str, Any], meta_path: Path) -> None:
    """Write extractor metadata JSON cleanly without secrets."""
    tmp = meta_path.with_suffix(".tmp.json")
    tmp.write_text(json.dumps(meta, indent=2), encoding="utf-8")
    tmp.replace(meta_path)
    print(f"Saved metadata JSON    : {meta_path}")


def write_not_run_metadata(
    extractor_key: str,
    model_name: str,
    weights_source: Dict[str, Any],
    preprocessing: Dict[str, Any],
    embed_dim: int,
    status: str,
    reason: str,
    is_control: bool = False,
) -> Dict[str, Any]:
    """Record a structured 'not run' or 'failed' metadata JSON without substituting another model."""
    meta_path = WORK_DIR / f"cached_dicom_features_{extractor_key}_metadata.json"
    safe_reason = sanitize_message(reason)
    meta = {
        "extractor_key": extractor_key,
        "status": status,
        "not_run_reason": safe_reason,
        "is_control": is_control,
        "model_name": model_name,
        "weights_source": weights_source,
        "preprocessing": preprocessing,
        "embedding_size": embed_dim,
        "column_prefix": f"{extractor_key}_emb_",
        "output_csv": str(WORK_DIR / f"cached_dicom_features_{extractor_key}.csv"),
        "patients_target_count": len(TARGET_PATIENTS),
        "patients_processed_count": 0,
        "patients_newly_processed_this_run": 0,
        "patients_skipped_count": 0,
        "skipped_patients": [],
        "timing_test_one_patient_sec": None,
        "estimated_total_time_min": None,
        "runtime_seconds": 0.0,
        "runtime_minutes": 0.0,
        "peak_gpu_memory_mb": (
            round(torch.cuda.max_memory_allocated() / (1024 ** 2), 2)
            if DEVICE.type == "cuda"
            else 0.0
        ),
        "peak_gpu_memory_reserved_mb": (
            round(torch.cuda.max_memory_reserved() / (1024 ** 2), 2)
            if DEVICE.type == "cuda"
            else 0.0
        ),
        "validation_checks": None,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "environment": {
            "torch_version": torch.__version__,
            "torchvision_version": torchvision.__version__,
            "device": str(DEVICE),
            "gpu_name": GPU_NAME,
            "gpu_total_mem_gb": GPU_TOTAL_MEM_GB,
        },
    }
    write_metadata_json(meta, meta_path)
    EXTRACTOR_RUN_REGISTRY[extractor_key] = meta
    return meta


def run_resumable_extractor(
    extractor_key: str,
    model_name: str,
    embed_dim: int,
    extract_patient_fn: Callable[
        [List[Tuple[np.ndarray, Tuple[float, float], float, float]], Optional[float]],
        np.ndarray,
    ],
    weights_source: Dict[str, Any],
    preprocessing: Dict[str, Any],
    is_control: bool = False,
) -> Dict[str, Any]:
    """
    Standard resumable runner shared by every extractor:
      1. Loads any existing partial CSV to resume on restart.
      2. Runs a one-patient timing test and prints estimated total time.
      3. Loops over all 175 target patients, skips and logs failures, checkpoints every 10 patients.
      4. Validates output CSV (count, file size, first 3 rows, NaN & constant columns) and writes metadata JSON.
    """
    output_csv = WORK_DIR / f"cached_dicom_features_{extractor_key}.csv"
    meta_path = WORK_DIR / f"cached_dicom_features_{extractor_key}_metadata.json"
    emb_cols = [f"{extractor_key}_emb_{i}" for i in range(embed_dim)]
    expected_cols = ["Patient"] + emb_cols

    completed_rows: Dict[str, Dict[str, Any]] = {}
    if output_csv.exists():
        df_prev = pd.read_csv(output_csv)
        if list(df_prev.columns) == expected_cols:
            for _, row in df_prev.iterrows():
                pid = str(row["Patient"])
                if pid in TARGET_PATIENTS:
                    completed_rows[pid] = row.to_dict()
            print(f"[{extractor_key}] Resuming from checkpoint: {len(completed_rows)}/{len(TARGET_PATIENTS)} patients already saved.")
        else:
            print(f"[{extractor_key}] Existing CSV schema mismatch; starting fresh -> {output_csv}")
    else:
        print(f"[{extractor_key}] No prior checkpoint found; starting fresh -> {output_csv}")

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()

    skipped_patients: List[Dict[str, str]] = []
    newly_processed = 0
    t_run_start = time.time()

    # --- Step 1: Mandatory one-patient timing test ---
    remaining_before_timing = [pid for pid in TARGET_PATIENTS if pid not in completed_rows]
    timing_pid = remaining_before_timing[0] if remaining_before_timing else TARGET_PATIENTS[0]
    t_one_start = time.time()
    timing_slices, timing_dz, _ = load_patient_sorted_slices(timing_pid)
    timing_vec = np.asarray(extract_patient_fn(timing_slices, timing_dz), dtype=np.float32).reshape(-1)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    t_one_sec = time.time() - t_one_start

    if timing_vec.shape != (embed_dim,):
        raise RuntimeError(
            f"[{extractor_key}] Embedding shape mismatch on timing patient {timing_pid}: "
            f"expected ({embed_dim},), got {timing_vec.shape}"
        )
    if not np.all(np.isfinite(timing_vec)):
        raise RuntimeError(f"[{extractor_key}] Non-finite embedding values on timing patient {timing_pid}")

    n_to_estimate = len(remaining_before_timing) if remaining_before_timing else len(TARGET_PATIENTS)
    est_remaining_min = (t_one_sec * n_to_estimate) / 60.0
    est_full_175_min = (t_one_sec * len(TARGET_PATIENTS)) / 60.0
    peak_one_mb = (
        torch.cuda.max_memory_allocated() / (1024 ** 2) if DEVICE.type == "cuda" else 0.0
    )
    print(
        f"[{extractor_key}] ONE-PATIENT TIMING TEST ({timing_pid}, {len(timing_slices)} slices): "
        f"{t_one_sec:.2f}s | Peak GPU: {peak_one_mb:.1f} MB | "
        f"Est. remaining ({n_to_estimate} pts): {est_remaining_min:.2f} min | "
        f"Est. full 175 pts: {est_full_175_min:.2f} min"
    )

    if timing_pid not in completed_rows:
        row_dict = {"Patient": timing_pid}
        for i, val in enumerate(timing_vec):
            row_dict[emb_cols[i]] = round(float(val), EMBED_ROUND_DECIMALS)
        completed_rows[timing_pid] = row_dict
        newly_processed += 1
        save_embedding_checkpoint(completed_rows, expected_cols, output_csv)

    # --- Step 2: Full resumable loop with 10-patient checkpointing ---
    pbar = tqdm(TARGET_PATIENTS, desc=f"Extracting [{extractor_key}]", unit="patient")
    for pid in pbar:
        if pid in completed_rows:
            pbar.set_postfix_str(f"{pid}: cached")
            continue

        t0 = time.time()
        try:
            sorted_slices, median_dz, slice_errs = load_patient_sorted_slices(pid)
            vec = np.asarray(extract_patient_fn(sorted_slices, median_dz), dtype=np.float32).reshape(-1)
            if vec.shape != (embed_dim,):
                raise RuntimeError(f"Expected ({embed_dim},) embedding, got {vec.shape}")
            if not np.all(np.isfinite(vec)):
                raise RuntimeError("Embedding contains NaN or Inf values")

            row_dict = {"Patient": pid}
            for i, val in enumerate(vec):
                row_dict[emb_cols[i]] = round(float(val), EMBED_ROUND_DECIMALS)
            completed_rows[pid] = row_dict
            newly_processed += 1

            dt = time.time() - t0
            pbar.set_postfix_str(
                f"{pid}: {len(sorted_slices)}sl in {dt:.1f}s"
                + (f" ({slice_errs} bad sl)" if slice_errs else "")
            )

            if newly_processed % CHECKPOINT_EVERY == 0:
                save_embedding_checkpoint(completed_rows, expected_cols, output_csv)
                tqdm.write(
                    f"[{extractor_key}] [CHECKPOINT] Saved {len(completed_rows)}/{len(TARGET_PATIENTS)} patients -> {output_csv.name}"
                )

        except torch.cuda.OutOfMemoryError as oom_exc:
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()
            reason = sanitize_message(f"CUDA OutOfMemoryError: {oom_exc}")
            skipped_patients.append({"patient": pid, "reason": reason})
            tqdm.write(f"[{extractor_key}] [SKIP-OOM] {pid}: {reason}")
        except Exception as exc:
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()
            reason = sanitize_message(f"{type(exc).__name__}: {exc}")
            skipped_patients.append({"patient": pid, "reason": reason})
            tqdm.write(f"[{extractor_key}] [SKIP] {pid}: {reason}")

    # Final atomic save
    save_embedding_checkpoint(completed_rows, expected_cols, output_csv)
    runtime_sec = time.time() - t_run_start
    peak_gpu_alloc_mb = (
        round(torch.cuda.max_memory_allocated() / (1024 ** 2), 2)
        if DEVICE.type == "cuda"
        else 0.0
    )
    peak_gpu_res_mb = (
        round(torch.cuda.max_memory_reserved() / (1024 ** 2), 2)
        if DEVICE.type == "cuda"
        else 0.0
    )

    val_checks = validate_and_summarize_csv(
        output_csv=output_csv,
        extractor_key=extractor_key,
        embed_dim=embed_dim,
        skipped_patients=skipped_patients,
    )

    meta = {
        "extractor_key": extractor_key,
        "status": "completed",
        "not_run_reason": None,
        "is_control": is_control,
        "model_name": model_name,
        "weights_source": weights_source,
        "preprocessing": preprocessing,
        "embedding_size": embed_dim,
        "column_prefix": f"{extractor_key}_emb_",
        "output_csv": str(output_csv),
        "patients_target_count": len(TARGET_PATIENTS),
        "patients_processed_count": len(completed_rows),
        "patients_newly_processed_this_run": newly_processed,
        "patients_skipped_count": len(skipped_patients),
        "skipped_patients": skipped_patients,
        "timing_test_one_patient_sec": round(t_one_sec, 3),
        "estimated_total_time_min": round(est_full_175_min, 2),
        "runtime_seconds": round(runtime_sec, 2),
        "runtime_minutes": round(runtime_sec / 60.0, 2),
        "peak_gpu_memory_mb": peak_gpu_alloc_mb,
        "peak_gpu_memory_reserved_mb": peak_gpu_res_mb,
        "validation_checks": val_checks,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "environment": {
            "torch_version": torch.__version__,
            "torchvision_version": torchvision.__version__,
            "device": str(DEVICE),
            "gpu_name": GPU_NAME,
            "gpu_total_mem_gb": GPU_TOTAL_MEM_GB,
        },
    }
    write_metadata_json(meta, meta_path)
    EXTRACTOR_RUN_REGISTRY[extractor_key] = meta
    return meta

---
## Section 1 — MedicalNet (Med3D): 3D ResNet-18 (`med3d`)

- **Model & Weights:** 3D ResNet-18 pretrained on 23 3D medical datasets (`resnet_18_23dataset.pth` from Hugging Face mirror `TencentMedicalNet/MedicalNet-Resnet18`), using exact `Tencent/MedicalNet/models/resnet.py` architecture (`shortcut_type='A'`, dilated `layer3` with `dilation=2`, dilated `layer4` with `dilation=4`, no segmentation decoder head, global average pooling `AdaptiveAvgPool3d((1, 1, 1))` -> **512-D** embedding).
- **Strict Weight Verification:** Strips `module.` (`DataParallel`) prefix, compares keys directly against the model's own `state_dict` keys (ignoring BatchNorm `num_batches_tracked` differences), prints actual counts, and fails loudly on any real key mismatch.
- **Preprocessing (per `Tencent/MedicalNet` `setting.py` & Chen et al., arXiv:1904.00625):**
  1. Load axial slices with `load_dicom_as_hu`, sort by Z-position, segment lungs per slice with `segment_lungs_hu`, set non-lung voxels to `-1000.0` HU.
  2. Stack into a 3D `(D, H, W)` volume and resample to fixed shape **`(56, 448, 448)`** (matching `Tencent/MedicalNet/setting.py` defaults `input_D=56, input_H=448, input_W=448`) via 3D trilinear interpolation (`align_corners=False`).
  3. Apply Med3D foreground intensity normalization: truncate lung voxels to the **0.5th–99.5th percentile**, z-score standardize `(vol - mean) / std` using foreground lung statistics, and set masked-out non-lung background deterministically to `0.0` (avoiding training-time random Gaussian noise so inference embeddings are deterministic).
- **Output:** `/kaggle/working/cached_dicom_features_med3d.csv` (`Patient` + `med3d_emb_0..511`) and `/kaggle/working/cached_dicom_features_med3d_metadata.json`.

In [ ]:
# Cell 6 (Section 1): MedicalNet (Med3D) 3D ResNet-18 extraction (isolated section)
from functools import partial
from huggingface_hub import hf_hub_download

MED3D_EXTRACTOR_KEY = "med3d"
MED3D_REPO_ID = "TencentMedicalNet/MedicalNet-Resnet18"
MED3D_FILENAME = "resnet_18_23dataset.pth"
MED3D_TARGET_SHAPE = (56, 448, 448)  # (input_D, input_H, input_W) from Tencent/MedicalNet/setting.py
MED3D_EMBED_DIM = 512

MED3D_PREPROC_SPEC = {
    "lung_masking": "segment_lungs_hu per slice; outside lung set to -1000.0 HU",
    "volume_resampling": "3D trilinear interpolation (align_corners=False) to fixed (D=56, H=448, W=448) per Tencent/MedicalNet setting.py",
    "intensity_normalization": "Med3D paper (arXiv:1904.00625): clip lung foreground to [0.5th, 99.5th] percentiles, z-score standardize by lung foreground mean/std, set outside-lung background deterministically to 0.0",
    "pooling": "3D Global Average Pooling (AdaptiveAvgPool3d((1, 1, 1))) after layer4 (before conv_seg head)",
    "batch_size": 1,
    "precision": "float32",
}


def _med3d_conv3x3x3(in_planes: int, out_planes: int, stride: int = 1, dilation: int = 1) -> nn.Conv3d:
    return nn.Conv3d(
        in_planes,
        out_planes,
        kernel_size=3,
        stride=stride,
        dilation=dilation,
        padding=dilation,
        bias=False,
    )


def _med3d_downsample_basic_block(x: torch.Tensor, planes: int, stride: int) -> torch.Tensor:
    """PyTorch 2.x-safe exact equivalent of downsample_basic_block in Tencent/MedicalNet/models/resnet.py (shortcut_type='A')."""
    out = F.avg_pool3d(x, kernel_size=1, stride=stride)
    zero_pads = torch.zeros(
        out.size(0),
        planes - out.size(1),
        out.size(2),
        out.size(3),
        out.size(4),
        dtype=out.dtype,
        device=out.device,
    )
    return torch.cat([out, zero_pads], dim=1)


class _Med3DBasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes: int, planes: int, stride: int = 1, dilation: int = 1, downsample=None):
        super().__init__()
        self.conv1 = _med3d_conv3x3x3(inplanes, planes, stride=stride, dilation=dilation)
        self.bn1 = nn.BatchNorm3d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = _med3d_conv3x3x3(planes, planes, dilation=dilation)
        self.bn2 = nn.BatchNorm3d(planes)
        self.downsample = downsample
        self.stride = stride
        self.dilation = dilation

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            residual = self.downsample(x)
        out = self.relu(out + residual)
        return out


class MedicalNetResNet18Backbone(nn.Module):
    """Exact Tencent/MedicalNet 3D ResNet-18 backbone (shortcut_type='A', dilated layer3/layer4) + Global Average Pooling."""

    def __init__(self, shortcut_type: str = "A"):
        super().__init__()
        self.inplanes = 64
        self.conv1 = nn.Conv3d(1, 64, kernel_size=7, stride=(2, 2, 2), padding=(3, 3, 3), bias=False)
        self.bn1 = nn.BatchNorm3d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool3d(kernel_size=(3, 3, 3), stride=2, padding=1)
        self.layer1 = self._make_layer(_Med3DBasicBlock, 64, 2, shortcut_type)
        self.layer2 = self._make_layer(_Med3DBasicBlock, 128, 2, shortcut_type, stride=2)
        self.layer3 = self._make_layer(_Med3DBasicBlock, 256, 2, shortcut_type, stride=1, dilation=2)
        self.layer4 = self._make_layer(_Med3DBasicBlock, 512, 2, shortcut_type, stride=1, dilation=4)
        self.avgpool = nn.AdaptiveAvgPool3d((1, 1, 1))

    def _make_layer(self, block, planes: int, blocks: int, shortcut_type: str, stride: int = 1, dilation: int = 1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            if shortcut_type == "A":
                downsample = partial(_med3d_downsample_basic_block, planes=planes * block.expansion, stride=stride)
            else:
                downsample = nn.Sequential(
                    nn.Conv3d(self.inplanes, planes * block.expansion, kernel_size=1, stride=stride, bias=False),
                    nn.BatchNorm3d(planes * block.expansion),
                )
        layers = [block(self.inplanes, planes, stride=stride, dilation=dilation, downsample=downsample)]
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes, dilation=dilation))
        return nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        return torch.flatten(x, 1)


def preprocess_volume_for_med3d(
    sorted_slices: List[Tuple[np.ndarray, Tuple[float, float], float, float]],
) -> torch.Tensor:
    """Build lung-masked 3D volume, resample to (56, 448, 448), and apply Med3D percentile + z-score normalization."""
    masked_slices = []
    mask_slices = []
    for hu_img, _, _, _ in sorted_slices:
        mask = segment_lungs_hu(hu_img)
        masked_hu = np.where(mask, hu_img, -1000.0).astype(np.float32)
        masked_slices.append(masked_hu)
        mask_slices.append(mask.astype(np.float32))

    vol_dhw = stack_slices_consistent_hw(masked_slices, mode="bilinear")
    mask_dhw = stack_slices_consistent_hw(mask_slices, mode="nearest")

    vol_t = torch.from_numpy(vol_dhw).unsqueeze(0).unsqueeze(0)   # (1, 1, D, H, W)
    mask_t = torch.from_numpy(mask_dhw).unsqueeze(0).unsqueeze(0) # (1, 1, D, H, W)

    vol_res = F.interpolate(vol_t, size=MED3D_TARGET_SHAPE, mode="trilinear", align_corners=False)[0, 0].numpy()
    mask_res = F.interpolate(mask_t, size=MED3D_TARGET_SHAPE, mode="nearest")[0, 0].numpy() > 0.5

    fg_voxels = vol_res[mask_res]
    if fg_voxels.size < 16:
        fg_voxels = vol_res[vol_res > -999.5]
    if fg_voxels.size < 16:
        fg_voxels = vol_res.ravel()

    p_lo, p_hi = np.percentile(fg_voxels, [0.5, 99.5])
    vol_clipped = np.clip(vol_res, float(p_lo), float(p_hi))
    fg_clipped = vol_clipped[mask_res] if mask_res.any() else vol_clipped.ravel()
    mean_v = float(np.mean(fg_clipped))
    std_v = float(np.std(fg_clipped))
    if not np.isfinite(std_v) or std_v < 1e-6:
        std_v = 1.0

    vol_norm = ((vol_clipped - mean_v) / std_v).astype(np.float32)
    if mask_res.any():
        vol_norm[~mask_res] = 0.0
    return torch.from_numpy(vol_norm).unsqueeze(0).unsqueeze(0)  # (1, 1, 56, 448, 448)


med3d_model = None
med3d_weights_meta: Dict[str, Any] = {
    "repo_id": MED3D_REPO_ID,
    "filename": MED3D_FILENAME,
    "architecture_ref": "Tencent/MedicalNet/models/resnet.py (resnet18, shortcut_type='A')",
}

try:
    print(f"Downloading {MED3D_REPO_ID}/{MED3D_FILENAME} ...")
    med3d_commit = get_hf_commit_sha(MED3D_REPO_ID, repo_type="model", token=HF_TOKEN)
    med3d_path = Path(
        hf_hub_download(repo_id=MED3D_REPO_ID, filename=MED3D_FILENAME, token=HF_TOKEN)
    )
    med3d_sha256 = compute_sha256(med3d_path)

    ckpt = safe_torch_load(med3d_path)
    raw_sd = ckpt["state_dict"] if isinstance(ckpt, dict) and "state_dict" in ckpt else ckpt
    if not isinstance(raw_sd, dict):
        raise RuntimeError(f"Expected state_dict mapping in {MED3D_FILENAME}, got {type(raw_sd).__name__}")

    clean_sd = {
        (k[len("module."):] if k.startswith("module.") else k): v
        for k, v in raw_sd.items()
    }

    med3d_model = MedicalNetResNet18Backbone(shortcut_type="A")
    model_sd_keys = set(med3d_model.state_dict().keys())
    ckpt_sd_keys = set(clean_sd.keys())

    # Compare strictly against the model's own state_dict keys, ignoring num_batches_tracked differences
    expected_core_keys = {k for k in model_sd_keys if not k.endswith(".num_batches_tracked")}
    loaded_core_keys = {k for k in ckpt_sd_keys if not k.endswith(".num_batches_tracked")}
    missing_k = sorted(expected_core_keys - loaded_core_keys)
    unexpected_k = sorted(loaded_core_keys - expected_core_keys)
    ignored_nbt_missing = sorted(
        k for k in (model_sd_keys - ckpt_sd_keys) if k.endswith(".num_batches_tracked")
    )
    ignored_nbt_unexpected = sorted(
        k for k in (ckpt_sd_keys - model_sd_keys) if k.endswith(".num_batches_tracked")
    )

    print(
        f"MedicalNet key check | model_state_dict_keys={len(model_sd_keys)} "
        f"(core={len(expected_core_keys)}) | checkpoint_keys={len(ckpt_sd_keys)} "
        f"(core={len(loaded_core_keys)}) | ignored_num_batches_tracked_diff="
        f"{len(ignored_nbt_missing) + len(ignored_nbt_unexpected)}"
    )

    if missing_k or unexpected_k:
        raise RuntimeError(
            "STRICT KEY MISMATCH loading MedicalNet ResNet-18 weights! "
            f"Model core keys={len(expected_core_keys)}, checkpoint core keys={len(loaded_core_keys)}. "
            f"Missing ({len(missing_k)}): {missing_k[:8]}; "
            f"Unexpected ({len(unexpected_k)}): {unexpected_k[:8]}"
        )

    filtered_sd = {k: v for k, v in clean_sd.items() if k in model_sd_keys}
    incompatible = med3d_model.load_state_dict(
        filtered_sd, strict=(len(ignored_nbt_missing) == 0)
    )
    bad_missing = [k for k in incompatible.missing_keys if not k.endswith(".num_batches_tracked")]
    bad_unexpected = [k for k in incompatible.unexpected_keys if not k.endswith(".num_batches_tracked")]
    if bad_missing or bad_unexpected:
        raise RuntimeError(
            f"STRICT LOAD FAILURE loading MedicalNet ResNet-18 weights: "
            f"missing={bad_missing[:8]}, unexpected={bad_unexpected[:8]}"
        )
    num_loaded_keys = len(filtered_sd)
    med3d_model.eval()
    for p in med3d_model.parameters():
        p.requires_grad = False
    med3d_model = med3d_model.to(DEVICE)
    del ckpt, raw_sd, clean_sd, filtered_sd
    gc.collect()

    med3d_weights_meta.update(
        {
            "repo_commit_sha": med3d_commit,
            "file_sha256": med3d_sha256,
            "file_size_bytes": int(med3d_path.stat().st_size),
            "strict_key_check_passed": True,
            "model_state_dict_key_count": len(model_sd_keys),
            "checkpoint_key_count": len(ckpt_sd_keys),
            "num_loaded_keys": num_loaded_keys,
            "ignored_num_batches_tracked_diff": len(ignored_nbt_missing) + len(ignored_nbt_unexpected),
        }
    )
    print(
        f"Loaded MedicalNet 3D ResNet-18 ({num_loaded_keys}/{len(model_sd_keys)} model keys loaded, "
        f"core={len(expected_core_keys)}/{len(expected_core_keys)}, SHA-256: {med3d_sha256[:16]}...) on {DEVICE}."
    )

    def _extract_med3d_patient(sorted_slices, _median_dz) -> np.ndarray:
        vol_tensor = preprocess_volume_for_med3d(sorted_slices).to(DEVICE, non_blocking=True)
        with torch.no_grad():
            emb = med3d_model(vol_tensor).squeeze(0).detach().cpu().numpy()
        return emb.astype(np.float32)

    run_resumable_extractor(
        extractor_key=MED3D_EXTRACTOR_KEY,
        model_name="MedicalNet (Med3D) 3D ResNet-18 (resnet_18_23dataset)",
        embed_dim=MED3D_EMBED_DIM,
        extract_patient_fn=_extract_med3d_patient,
        weights_source=med3d_weights_meta,
        preprocessing=MED3D_PREPROC_SPEC,
        is_control=False,
    )

except Exception as exc:
    err_msg = sanitize_message(f"{type(exc).__name__}: {exc}")
    print(f"[ERROR] Section 1 ({MED3D_EXTRACTOR_KEY}) failed and will NOT stop later sections: {err_msg}")
    write_not_run_metadata(
        extractor_key=MED3D_EXTRACTOR_KEY,
        model_name="MedicalNet (Med3D) 3D ResNet-18 (resnet_18_23dataset)",
        weights_source=med3d_weights_meta,
        preprocessing=MED3D_PREPROC_SPEC,
        embed_dim=MED3D_EMBED_DIM,
        status="failed",
        reason=err_msg,
        is_control=False,
    )
finally:
    if med3d_model is not None:
        del med3d_model
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

---
## Section 2 — RadImageNet: 2D ResNet-50 (`radimagenet`)

- **Model & Weights:** 2D ResNet-50 pretrained on 1.35M radiologic images (`Lab-Rasool/RadImageNet`, file `ResNet50.pt`, matching the official `BMEII-AI/RadImageNet` PyTorch `Backbone` state dictionary: `nn.Sequential(*list(resnet50(weights=None).children())[:9])` with **318** parameter/buffer keys `backbone.0.*` through `backbone.7.*` and global average pooling `backbone.8` outputting **2048-D** embeddings).
- **Strict Key Checking:** Verifies exact 318/318 key set equality before `load_state_dict(..., strict=True)`. Fails loudly on any missing, unexpected, or shape-mismatched key; never falls back to ImageNet or random weights.
- **Preprocessing:** Bit-for-bit identical per-slice recipe to `extract_pretrained_resnet18_slice_embedding` in `backend/src/imaging_progression.py`:
  - `segment_lungs_hu` lung mask -> outside lung set to `-1000.0` HU
  - Window `[-1000, 0]` HU -> `[0, 1]` -> `uint8` -> PIL resize `224x224` (`BILINEAR`) -> `/ 255.0`
  - 3-channel repeat + ImageNet normalization (`mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]`)
  - GPU-batched (`batch_size=64`) per-slice inference, mean-pooled across all slices into **2048** features.
- **Output:** `/kaggle/working/cached_dicom_features_radimagenet.csv` (`Patient` + `radimagenet_emb_0..2047`) and `/kaggle/working/cached_dicom_features_radimagenet_metadata.json`.

In [ ]:
# Cell 8 (Section 2): RadImageNet 2D ResNet-50 extraction with strict key verification (isolated section)
from huggingface_hub import hf_hub_download
from torchvision.models import resnet50

RAD_EXTRACTOR_KEY = "radimagenet"
RAD_REPO_ID = "Lab-Rasool/RadImageNet"
RAD_FILENAME = "ResNet50.pt"
RAD_EMBED_DIM = 2048

RAD_PREPROC_SPEC = {
    "lung_masking": "segment_lungs_hu per slice; outside lung set to -1000.0 HU (matches imaging_progression.py)",
    "hu_window": "[-1000, 0] HU linearly scaled to [0, 1]",
    "resize": "uint8 -> PIL 224x224 BILINEAR -> float32 / 255.0 (matches imaging_progression.py)",
    "normalization": "3-channel grayscale repeat + ImageNet mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]",
    "pooling": "AdaptiveAvgPool2d((1, 1)) per slice -> float64 mean-pool over all axial slices",
    "slice_batch_size": SLICE_BATCH_SIZE_2D,
    "precision": "float32",
}


class RadImageNetResNet50Backbone(nn.Module):
    """Exact BMEII-AI / Lab-Rasool RadImageNet PyTorch ResNet-50 Backbone (`children()[:9]`, 318 keys -> 2048-D)."""

    def __init__(self):
        super().__init__()
        base_model = resnet50(weights=None)
        self.backbone = nn.Sequential(*list(base_model.children())[:9])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.flatten(self.backbone(x), 1)


rad_model = None
rad_weights_meta: Dict[str, Any] = {
    "repo_id": RAD_REPO_ID,
    "filename": RAD_FILENAME,
    "upstream_reference": "BMEII-AI/RadImageNet pytorch_example.ipynb (Backbone = nn.Sequential(*children[:9]))",
}

try:
    print(f"Downloading {RAD_REPO_ID}/{RAD_FILENAME} ...")
    rad_commit = get_hf_commit_sha(RAD_REPO_ID, repo_type="model", token=HF_TOKEN)
    rad_path = Path(
        hf_hub_download(repo_id=RAD_REPO_ID, filename=RAD_FILENAME, token=HF_TOKEN)
    )
    rad_sha256 = compute_sha256(rad_path)

    raw_ckpt = safe_torch_load(rad_path)
    if not isinstance(raw_ckpt, dict):
        raise RuntimeError(
            f"STRICT LOAD FAILURE: Expected a state_dict mapping in {RAD_FILENAME}, got {type(raw_ckpt).__name__}."
        )
    if "state_dict" in raw_ckpt and isinstance(raw_ckpt["state_dict"], dict):
        rad_sd = raw_ckpt["state_dict"]
    else:
        rad_sd = raw_ckpt

    rad_model = RadImageNetResNet50Backbone()
    expected_keys = set(rad_model.state_dict().keys())
    loaded_keys = set(rad_sd.keys())
    missing_keys = sorted(expected_keys - loaded_keys)
    unexpected_keys = sorted(loaded_keys - expected_keys)

    if missing_keys or unexpected_keys or len(loaded_keys) != len(expected_keys):
        raise RuntimeError(
            "STRICT KEY MISMATCH loading RadImageNet ResNet50 weights! "
            f"Expected {len(expected_keys)} keys (e.g. {sorted(expected_keys)[:5]}), "
            f"got {len(loaded_keys)} keys (e.g. {sorted(loaded_keys)[:5]}). "
            f"Missing ({len(missing_keys)}): {missing_keys[:8]}; "
            f"Unexpected ({len(unexpected_keys)}): {unexpected_keys[:8]}"
        )

    incompatible = rad_model.load_state_dict(rad_sd, strict=True)
    if incompatible.missing_keys or incompatible.unexpected_keys:
        raise RuntimeError(
            f"STRICT LOAD FAILURE: missing={incompatible.missing_keys}, unexpected={incompatible.unexpected_keys}"
        )

    rad_model.eval()
    for p in rad_model.parameters():
        p.requires_grad = False
    rad_model = rad_model.to(DEVICE)
    del raw_ckpt, rad_sd
    gc.collect()

    rad_weights_meta.update(
        {
            "repo_commit_sha": rad_commit,
            "file_sha256": rad_sha256,
            "file_size_bytes": int(rad_path.stat().st_size),
            "strict_key_check_passed": True,
            "num_loaded_keys": len(expected_keys),
        }
    )
    print(
        f"Loaded RadImageNet 2D ResNet-50 strictly ({len(expected_keys)}/{len(expected_keys)} keys, "
        f"SHA-256: {rad_sha256[:16]}...) on {DEVICE}."
    )

    def _extract_radimagenet_patient(sorted_slices, _median_dz) -> np.ndarray:
        return extract_2d_backbone_patient_embedding(
            sorted_slices=sorted_slices,
            model=rad_model,
            embed_dim=RAD_EMBED_DIM,
            batch_size=SLICE_BATCH_SIZE_2D,
        )

    run_resumable_extractor(
        extractor_key=RAD_EXTRACTOR_KEY,
        model_name="RadImageNet 2D ResNet-50 (Lab-Rasool/RadImageNet ResNet50.pt)",
        embed_dim=RAD_EMBED_DIM,
        extract_patient_fn=_extract_radimagenet_patient,
        weights_source=rad_weights_meta,
        preprocessing=RAD_PREPROC_SPEC,
        is_control=False,
    )

except Exception as exc:
    err_msg = sanitize_message(f"{type(exc).__name__}: {exc}")
    print(f"[ERROR] Section 2 ({RAD_EXTRACTOR_KEY}) failed and will NOT stop later sections: {err_msg}")
    write_not_run_metadata(
        extractor_key=RAD_EXTRACTOR_KEY,
        model_name="RadImageNet 2D ResNet-50 (Lab-Rasool/RadImageNet ResNet50.pt)",
        weights_source=rad_weights_meta,
        preprocessing=RAD_PREPROC_SPEC,
        embed_dim=RAD_EMBED_DIM,
        status="failed",
        reason=err_msg,
        is_control=False,
    )
finally:
    if rad_model is not None:
        del rad_model
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

---
## Section 3 — CT-CLIP: 3D CT-ViT Image Encoder (`ctclip`)

- **Model & Weights:**
  - Architecture repo: `https://github.com/ibrahimethemhamamci/CT-CLIP` (`transformer_maskgit.CTViT` with `dim=512, codebook_size=8192, image_size=480, patch_size=20, temporal_patch_size=10, spatial_depth=4, temporal_depth=4, dim_head=32, heads=8` + `to_visual_latent = nn.Linear(294912, 512, bias=False)` + L2 normalization, reproducing `CT_CLIP/ct_clip/ct_clip.py` lines 715–771 and `scripts/forward_data.py` verbatim).
  - Weights: Gated Hugging Face dataset `ibrahimhamamci/CT-RATE`, file `models/CT-CLIP-Related/CT-CLIP_v2.pt` (authenticated with Kaggle Secret `HF_TOKEN`).
  - Exact dependency pinning: `vector-quantize-pytorch==1.1.2` (required by `CT-CLIP_v2.pt` so `vq._codebook.embed_avg` loads strictly).
- **Preprocessing (reproduced verbatim from `ibrahimethemhamamci/CT-CLIP/scripts/data_inference_nii.py`):**
  1. Load full-chest HU slices via `load_dicom_as_hu` (already applies `RescaleSlope` & `RescaleIntercept`; **no double rescaling** and **no lung masking**, matching CT-CLIP pretraining on full chest CT).
  2. Sort slices by Z-position; stack into `(D, H, W)` with spacing `(z_spacing, row_spacing, col_spacing)`.
  3. Clip HU to **`[-1000, 1000]`**.
  4. 3D trilinear resample (`align_corners=False`) from `(z_spacing, row_spacing, col_spacing)` to target spacing **`(1.5, 0.75, 0.75)` mm**.
  5. Divide by `1000.0` so voxel intensities lie in **`[-1.0, +1.0]`**.
  6. Center crop or pad with **`value = -1.0`** (`-1000 HU`) to exact shape `(H=480, W=480, D=240)`, then permute to **`(1, 1, 240, 480, 480)`**.
- **Serving & Failure Policy:** Runs at **`batch_size = 1`** with **`fp16` CUDA autocast** (falling back to `fp32` only on non-OOM autocast dtype incompatibility). If gated Hugging Face access, CUDA availability, weight loading, or GPU memory fails, this section records **`"status": "not run"`** with the exact reason in `cached_dicom_features_ctclip_metadata.json` and **never substitutes another model**.
- **Output:** `/kaggle/working/cached_dicom_features_ctclip.csv` (`Patient` + `ctclip_emb_0..511`) and `/kaggle/working/cached_dicom_features_ctclip_metadata.json`.

In [ ]:
# Cell 10 (Section 3): CT-CLIP 3D CT-ViT image encoder extraction (isolated section; records 'not run' on access/OOM failure)
from huggingface_hub import hf_hub_download

CTCLIP_EXTRACTOR_KEY = "ctclip"
CTCLIP_GITHUB_URL = "https://github.com/ibrahimethemhamamci/CT-CLIP.git"
CTCLIP_HF_REPO_ID = "ibrahimhamamci/CT-RATE"
CTCLIP_HF_FILENAME = "models/CT-CLIP-Related/CT-CLIP_v2.pt"
CTCLIP_EMBED_DIM = 512

# Exact target spacing & volume geometry from ibrahimethemhamamci/CT-CLIP/scripts/data_inference_nii.py
CTCLIP_TARGET_Z_SPACING = 1.5
CTCLIP_TARGET_X_SPACING = 0.75
CTCLIP_TARGET_Y_SPACING = 0.75
CTCLIP_HU_MIN = -1000.0
CTCLIP_HU_MAX = 1000.0
CTCLIP_TARGET_HWD = (480, 480, 240)  # (dh, dw, dd) -> permuted to (1, 1, 240, 480, 480)

CTCLIP_PREPROC_SPEC = {
    "upstream_script": "ibrahimethemhamamci/CT-CLIP/scripts/data_inference_nii.py (CTReportDatasetinfer.nii_img_to_tensor)",
    "lung_masking": "None (full-chest CT HU volume matching CT-CLIP pretraining)",
    "hu_clip": [-1000.0, 1000.0],
    "hu_scaling": "img_data / 1000.0 -> [-1.0, +1.0] float32",
    "target_spacing_mm_z_x_y": [CTCLIP_TARGET_Z_SPACING, CTCLIP_TARGET_X_SPACING, CTCLIP_TARGET_Y_SPACING],
    "resampling": "F.interpolate(..., mode='trilinear', align_corners=False) from (z_spacing, row_spacing, col_spacing) to (1.5, 0.75, 0.75) mm",
    "crop_pad_target_hwd": list(CTCLIP_TARGET_HWD),
    "pad_value": -1.0,
    "final_tensor_shape_b_c_d_h_w": [1, 1, 240, 480, 480],
    "pooling_and_projection": "Exact CTCLIP.forward: CTViT(..., return_encoded_tokens=True) -> mean(dim=1) -> flatten(294912) -> to_visual_latent(294912->512) -> L2 normalize",
    "batch_size": 1,
    "precision": "cuda_amp_fp16_autocast (fallback fp32 only on non-OOM dtype error)",
}


def preprocess_volume_for_ctclip(
    sorted_slices: List[Tuple[np.ndarray, Tuple[float, float], float, float]],
    median_dz: Optional[float],
) -> torch.Tensor:
    """
    Exact reproduction of `resize_array` + `nii_img_to_tensor` from
    `ibrahimethemhamamci/CT-CLIP/scripts/data_inference_nii.py` (lines 12-34 and 107-165),
    starting from HU slices decoded by `load_dicom_as_hu`.
    """
    hu_slices = [item[0] for item in sorted_slices]
    row_spacings = [float(item[1][0]) for item in sorted_slices if item[1][0] and item[1][0] > 0]
    col_spacings = [float(item[1][1]) for item in sorted_slices if item[1][1] and item[1][1] > 0]
    thicknesses = [float(item[2]) for item in sorted_slices if item[2] and item[2] > 0]

    row_spacing = float(np.median(row_spacings)) if row_spacings else CTCLIP_TARGET_X_SPACING
    col_spacing = float(np.median(col_spacings)) if col_spacings else CTCLIP_TARGET_Y_SPACING
    if median_dz is not None and 0.1 < median_dz <= 15.0:
        z_spacing = float(median_dz)
    elif thicknesses:
        z_spacing = float(np.median(thicknesses))
    else:
        z_spacing = CTCLIP_TARGET_Z_SPACING

    # Stack to (D, H, W) and clip HU to [-1000, 1000]
    vol_dhw = stack_slices_consistent_hw(hu_slices, mode="bilinear")
    vol_dhw = np.clip(vol_dhw, CTCLIP_HU_MIN, CTCLIP_HU_MAX).astype(np.float32)

    # Exact spacing-based trilinear resampling on (1, 1, D, H, W)
    tensor = torch.from_numpy(vol_dhw).unsqueeze(0).unsqueeze(0)
    current_spacing = (z_spacing, row_spacing, col_spacing)
    target_spacing = (CTCLIP_TARGET_Z_SPACING, CTCLIP_TARGET_X_SPACING, CTCLIP_TARGET_Y_SPACING)
    original_shape = tensor.shape[2:]
    new_shape = [
        max(1, int(original_shape[i] * (current_spacing[i] / target_spacing[i])))
        for i in range(3)
    ]
    resized_dhw = F.interpolate(tensor, size=new_shape, mode="trilinear", align_corners=False)[0, 0]

    # Match data_inference_nii.py lines 126-163 verbatim: (D', H', W') -> (H', W', D'), /1000, crop/pad (480, 480, 240) with -1
    tensor_hwd = (resized_dhw.permute(1, 2, 0) / 1000.0).to(torch.float32)
    h, w, d = tensor_hwd.shape
    dh, dw, dd = CTCLIP_TARGET_HWD

    h_start = max((h - dh) // 2, 0)
    h_end = min(h_start + dh, h)
    w_start = max((w - dw) // 2, 0)
    w_end = min(w_start + dw, w)
    d_start = max((d - dd) // 2, 0)
    d_end = min(d_start + dd, d)

    tensor_hwd = tensor_hwd[h_start:h_end, w_start:w_end, d_start:d_end]

    pad_h_before = (dh - tensor_hwd.size(0)) // 2
    pad_h_after = dh - tensor_hwd.size(0) - pad_h_before
    pad_w_before = (dw - tensor_hwd.size(1)) // 2
    pad_w_after = dw - tensor_hwd.size(1) - pad_w_before
    pad_d_before = (dd - tensor_hwd.size(2)) // 2
    pad_d_after = dd - tensor_hwd.size(2) - pad_d_before

    tensor_hwd = F.pad(
        tensor_hwd,
        (pad_d_before, pad_d_after, pad_w_before, pad_w_after, pad_h_before, pad_h_after),
        value=-1.0,
    )

    # Permute back to (D, H, W) = (240, 480, 480) and add (B=1, C=1) -> (1, 1, 240, 480, 480)
    tensor_bcdhw = tensor_hwd.permute(2, 0, 1).unsqueeze(0).unsqueeze(0).contiguous()
    assert tuple(tensor_bcdhw.shape) == (1, 1, 240, 480, 480), f"Unexpected CT-CLIP tensor shape: {tensor_bcdhw.shape}"
    return tensor_bcdhw


ctclip_vit = None
ctclip_proj = None
ctclip_weights_meta: Dict[str, Any] = {
    "github_repo": CTCLIP_GITHUB_URL,
    "hf_dataset_repo": CTCLIP_HF_REPO_ID,
    "filename": CTCLIP_HF_FILENAME,
}

try:
    if DEVICE.type != "cuda":
        raise RuntimeError(
            "CUDA GPU is required by upstream transformer_maskgit.ctvit (which hardcodes torch.device('cuda')), "
            "but no CUDA GPU is available."
        )
    if not HF_TOKEN:
        raise PermissionError(
            "HF_TOKEN Kaggle Secret is missing or empty; ibrahimhamamci/CT-RATE is a gated Hugging Face dataset."
        )

    # 1. Clone official CT-CLIP repo and install exact lightweight dependencies for transformer_maskgit
    ctclip_repo_dir = WORK_DIR / "repos" / "CT-CLIP"
    ctclip_repo_dir.parent.mkdir(parents=True, exist_ok=True)
    if not ctclip_repo_dir.exists():
        print(f"Cloning {CTCLIP_GITHUB_URL} ...")
        subprocess.check_call(["git", "clone", "--depth", "1", CTCLIP_GITHUB_URL, str(ctclip_repo_dir)])

    ctclip_git_commit = subprocess.check_output(
        ["git", "-C", str(ctclip_repo_dir), "rev-parse", "HEAD"], text=True
    ).strip()
    ctclip_weights_meta["github_commit_sha"] = ctclip_git_commit

    # Install exact vector-quantize-pytorch==1.1.2 + transformer_maskgit (--no-deps to preserve Kaggle's torch/transformers)
    print("Installing CT-CLIP image encoder dependencies (vector-quantize-pytorch==1.1.2, beartype, einops, ema-pytorch, nibabel, accelerate)...")
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "vector-quantize-pytorch==1.1.2",
            "beartype",
            "einops>=0.6",
            "ema-pytorch>=0.2.2",
            "nibabel",
            "accelerate",
        ]
    )
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--no-deps",
            "-e",
            str(ctclip_repo_dir / "transformer_maskgit"),
        ]
    )

    tmg_path = str(ctclip_repo_dir / "transformer_maskgit")
    if tmg_path not in sys.path:
        sys.path.insert(0, tmg_path)

    from transformer_maskgit import CTViT

    # 2. Download gated CT-CLIP_v2.pt checkpoint from ibrahimhamamci/CT-RATE
    print(f"Downloading gated checkpoint {CTCLIP_HF_REPO_ID}/{CTCLIP_HF_FILENAME} ...")
    ctrate_commit = get_hf_commit_sha(CTCLIP_HF_REPO_ID, repo_type="dataset", token=HF_TOKEN)
    ctclip_ckpt_path = Path(
        hf_hub_download(
            repo_id=CTCLIP_HF_REPO_ID,
            filename=CTCLIP_HF_FILENAME,
            repo_type="dataset",
            token=HF_TOKEN,
        )
    )
    ctclip_sha256 = compute_sha256(ctclip_ckpt_path)

    # 3. Instantiate exact CTViT image encoder + to_visual_latent projection and load strictly
    ctclip_vit = CTViT(
        dim=512,
        codebook_size=8192,
        image_size=480,
        patch_size=20,
        temporal_patch_size=10,
        spatial_depth=4,
        temporal_depth=4,
        dim_head=32,
        heads=8,
    )
    ctclip_proj = nn.Linear(294912, CTCLIP_EMBED_DIM, bias=False)

    ckpt_all = safe_torch_load(ctclip_ckpt_path)
    if not isinstance(ckpt_all, dict):
        raise RuntimeError(f"Expected dict state_dict in {CTCLIP_HF_FILENAME}, got {type(ckpt_all).__name__}")

    vit_sd = {
        k[len("visual_transformer."):]: v
        for k, v in ckpt_all.items()
        if k.startswith("visual_transformer.")
    }
    if not vit_sd:
        raise RuntimeError(
            f"No 'visual_transformer.*' keys found in {CTCLIP_HF_FILENAME}. Sample keys: {list(ckpt_all.keys())[:10]}"
        )
    if "to_visual_latent.weight" not in ckpt_all:
        raise RuntimeError(
            f"Missing 'to_visual_latent.weight' in {CTCLIP_HF_FILENAME}. Sample keys: {list(ckpt_all.keys())[:10]}"
        )
    proj_sd = {"weight": ckpt_all["to_visual_latent.weight"]}

    ctclip_vit.load_state_dict(vit_sd, strict=True)
    ctclip_proj.load_state_dict(proj_sd, strict=True)

    num_ctclip_keys = len(vit_sd) + len(proj_sd)
    del ckpt_all, vit_sd, proj_sd
    gc.collect()

    ctclip_vit.eval()
    ctclip_proj.eval()
    for p in ctclip_vit.parameters():
        p.requires_grad = False
    for p in ctclip_proj.parameters():
        p.requires_grad = False

    ctclip_vit = ctclip_vit.to(DEVICE)
    ctclip_proj = ctclip_proj.to(DEVICE)

    # Verify fp16 autocast compatibility on a first-patient probe; fall back to fp32 only on non-OOM dtype error
    ctclip_use_fp16 = True
    _probe_slices, _probe_dz, _ = load_patient_sorted_slices(TARGET_PATIENTS[0])
    _probe_vol = preprocess_volume_for_ctclip(_probe_slices, _probe_dz).to(DEVICE)
    _tok = None
    try:
        with torch.no_grad():
            with torch.amp.autocast("cuda", dtype=torch.float16, enabled=True):
                _tok = ctclip_vit(_probe_vol, return_encoded_tokens=True)
            _ = ctclip_proj(torch.mean(_tok.float(), dim=1).reshape(1, -1))
        CTCLIP_PREPROC_SPEC["precision"] = "cuda_amp_fp16_autocast"
    except torch.cuda.OutOfMemoryError:
        raise
    except RuntimeError as dtype_exc:
        if "out of memory" in str(dtype_exc).lower():
            raise
        print(f"  [INFO] fp16 autocast incompatible ({dtype_exc}); switching CT-CLIP to fp32 (batch_size=1).")
        ctclip_use_fp16 = False
        CTCLIP_PREPROC_SPEC["precision"] = "float32 (batch_size=1)"
    finally:
        if _tok is not None:
            del _tok
        del _probe_vol, _probe_slices
        torch.cuda.empty_cache()

    ctclip_weights_meta.update(
        {
            "repo_commit_sha": ctrate_commit,
            "file_sha256": ctclip_sha256,
            "file_size_bytes": int(ctclip_ckpt_path.stat().st_size),
            "strict_key_check_passed": True,
            "num_loaded_keys": num_ctclip_keys,
        }
    )
    print(
        f"Loaded CT-CLIP CTViT + to_visual_latent strictly ({num_ctclip_keys} keys, "
        f"SHA-256: {ctclip_sha256[:16]}..., precision={CTCLIP_PREPROC_SPEC['precision']}) on {DEVICE}."
    )

    def _extract_ctclip_patient(sorted_slices, median_dz) -> np.ndarray:
        vol_tensor = preprocess_volume_for_ctclip(sorted_slices, median_dz).to(DEVICE, non_blocking=True)
        with torch.no_grad():
            with torch.amp.autocast("cuda", dtype=torch.float16, enabled=ctclip_use_fp16):
                enc_tokens = ctclip_vit(vol_tensor, return_encoded_tokens=True)  # (1, 24, 24, 24, 512)
            # Exact pooling + visual latent projection + L2 norm from CT_CLIP/ct_clip/ct_clip.py lines 724-771
            enc_pooled = torch.mean(enc_tokens.float(), dim=1)                  # (1, 24, 24, 512)
            enc_flat = enc_pooled.reshape(enc_pooled.shape[0], -1)              # (1, 294912)
            latent = ctclip_proj(enc_flat)                                      # (1, 512)
            emb = F.normalize(latent, dim=-1).squeeze(0).detach().cpu().numpy() # (512,)
        del vol_tensor, enc_tokens, enc_pooled, enc_flat, latent
        return emb.astype(np.float32)

    run_resumable_extractor(
        extractor_key=CTCLIP_EXTRACTOR_KEY,
        model_name="CT-CLIP 3D CT-ViT Image Encoder + Visual Latent Projection (CT-CLIP_v2.pt)",
        embed_dim=CTCLIP_EMBED_DIM,
        extract_patient_fn=_extract_ctclip_patient,
        weights_source=ctclip_weights_meta,
        preprocessing=CTCLIP_PREPROC_SPEC,
        is_control=False,
    )

except Exception as exc:
    err_msg = sanitize_message(f"{type(exc).__name__}: {exc}")
    print(
        f"[NOT RUN] Section 3 ({CTCLIP_EXTRACTOR_KEY}) could not run (no substitute model used): {err_msg}"
    )
    write_not_run_metadata(
        extractor_key=CTCLIP_EXTRACTOR_KEY,
        model_name="CT-CLIP 3D CT-ViT Image Encoder + Visual Latent Projection (CT-CLIP_v2.pt)",
        weights_source=ctclip_weights_meta,
        preprocessing=CTCLIP_PREPROC_SPEC,
        embed_dim=CTCLIP_EMBED_DIM,
        status="not run",
        reason=err_msg,
        is_control=False,
    )
finally:
    if ctclip_vit is not None:
        del ctclip_vit
    if ctclip_proj is not None:
        del ctclip_proj
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

---
## Section 4 (Optional Extra Control) — ImageNet 2D ResNet-50 (`resnet50_imagenet_control`)

> **EXTRA CONTROL MODEL:** This section is an **optional architectural control** (set `RUN_IMAGENET_RESNET50_CONTROL = False` to skip). Because the PulmoSight baseline uses ImageNet **ResNet-18** (512-D) while RadImageNet uses **ResNet-50** (2048-D), extracting ImageNet **ResNet-50** on the exact same per-slice recipe isolates the effect of **RadImageNet medical pretraining** from **ResNet-50 backbone capacity**.

- **Model & Weights:** `torchvision.models.resnet50(weights=ResNet50_Weights.DEFAULT)` with `fc = nn.Identity()` (**2048-D**).
- **Preprocessing:** Bit-for-bit identical per-slice lung-masked `-1000..0 HU`, `224x224` PIL `BILINEAR`, ImageNet-normalized recipe as `backend/src/imaging_progression.py` and Section 2 (RadImageNet).
- **Output:** `/kaggle/working/cached_dicom_features_resnet50_imagenet_control.csv` (`Patient` + `resnet50_imagenet_control_emb_0..2047`) and `/kaggle/working/cached_dicom_features_resnet50_imagenet_control_metadata.json`.

In [ ]:
# Cell 12 (Section 4 — OPTIONAL EXTRA CONTROL): ImageNet 2D ResNet-50 on the exact same per-slice recipe
from torchvision.models import ResNet50_Weights, resnet50

RUN_IMAGENET_RESNET50_CONTROL = True

CTRL_EXTRACTOR_KEY = "resnet50_imagenet_control"
CTRL_EMBED_DIM = 2048
CTRL_PREPROC_SPEC = {
    "role": "EXTRA CONTROL (isolates ResNet-50 architecture vs. RadImageNet medical pretraining)",
    "lung_masking": "segment_lungs_hu per slice; outside lung set to -1000.0 HU (matches imaging_progression.py)",
    "hu_window": "[-1000, 0] HU linearly scaled to [0, 1]",
    "resize": "uint8 -> PIL 224x224 BILINEAR -> float32 / 255.0 (matches imaging_progression.py)",
    "normalization": "3-channel grayscale repeat + ImageNet mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]",
    "pooling": "AdaptiveAvgPool2d((1, 1)) per slice -> float64 mean-pool over all axial slices",
    "slice_batch_size": SLICE_BATCH_SIZE_2D,
    "precision": "float32",
}

ctrl_model = None
ctrl_weights_enum = ResNet50_Weights.DEFAULT
ctrl_weights_meta: Dict[str, Any] = {
    "source": "torchvision.models.resnet50",
    "weights_enum": str(ctrl_weights_enum),
    "weights_url": getattr(ctrl_weights_enum, "url", None),
    "is_extra_control": True,
}

if not RUN_IMAGENET_RESNET50_CONTROL:
    print(f"[SKIPPED] Optional extra control ({CTRL_EXTRACTOR_KEY}) disabled by RUN_IMAGENET_RESNET50_CONTROL=False.")
    write_not_run_metadata(
        extractor_key=CTRL_EXTRACTOR_KEY,
        model_name="EXTRA CONTROL: ImageNet 2D ResNet-50 (torchvision ResNet50_Weights.DEFAULT)",
        weights_source=ctrl_weights_meta,
        preprocessing=CTRL_PREPROC_SPEC,
        embed_dim=CTRL_EMBED_DIM,
        status="not run",
        reason="Disabled by user toggle RUN_IMAGENET_RESNET50_CONTROL = False",
        is_control=True,
    )
else:
    try:
        ctrl_model = resnet50(weights=ctrl_weights_enum)
        ctrl_model.fc = nn.Identity()
        ctrl_model.eval()
        for p in ctrl_model.parameters():
            p.requires_grad = False
        ctrl_model = ctrl_model.to(DEVICE)

        # Compute SHA-256 of the downloaded torchvision checkpoint in torch.hub cache
        cached_ckpt_name = Path(str(ctrl_weights_enum.url)).name if getattr(ctrl_weights_enum, "url", None) else ""
        cached_ckpt_path = Path(torch.hub.get_dir()) / "checkpoints" / cached_ckpt_name
        if cached_ckpt_name and cached_ckpt_path.exists():
            ctrl_weights_meta["file_sha256"] = compute_sha256(cached_ckpt_path)
            ctrl_weights_meta["file_size_bytes"] = int(cached_ckpt_path.stat().st_size)
        ctrl_weights_meta["strict_key_check_passed"] = True

        print(
            f"Loaded EXTRA CONTROL ImageNet 2D ResNet-50 ({ctrl_weights_enum}, "
            f"SHA-256: {str(ctrl_weights_meta.get('file_sha256', 'n/a'))[:16]}...) on {DEVICE}."
        )

        def _extract_ctrl_patient(sorted_slices, _median_dz) -> np.ndarray:
            return extract_2d_backbone_patient_embedding(
                sorted_slices=sorted_slices,
                model=ctrl_model,
                embed_dim=CTRL_EMBED_DIM,
                batch_size=SLICE_BATCH_SIZE_2D,
            )

        run_resumable_extractor(
            extractor_key=CTRL_EXTRACTOR_KEY,
            model_name="EXTRA CONTROL: ImageNet 2D ResNet-50 (torchvision ResNet50_Weights.DEFAULT)",
            embed_dim=CTRL_EMBED_DIM,
            extract_patient_fn=_extract_ctrl_patient,
            weights_source=ctrl_weights_meta,
            preprocessing=CTRL_PREPROC_SPEC,
            is_control=True,
        )

    except Exception as exc:
        err_msg = sanitize_message(f"{type(exc).__name__}: {exc}")
        print(f"[ERROR] Section 4 ({CTRL_EXTRACTOR_KEY}) failed: {err_msg}")
        write_not_run_metadata(
            extractor_key=CTRL_EXTRACTOR_KEY,
            model_name="EXTRA CONTROL: ImageNet 2D ResNet-50 (torchvision ResNet50_Weights.DEFAULT)",
            weights_source=ctrl_weights_meta,
            preprocessing=CTRL_PREPROC_SPEC,
            embed_dim=CTRL_EMBED_DIM,
            status="failed",
            reason=err_msg,
            is_control=True,
        )
    finally:
        if ctrl_model is not None:
            del ctrl_model
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

In [ ]:
# Cell 13: Final consolidated summary of all generated CSVs and metadata JSONs in /kaggle/working
ALL_EXTRACTORS = [
    ("med3d", 512),
    ("radimagenet", 2048),
    ("ctclip", 512),
    ("resnet50_imagenet_control", 2048),
]

summary_rows = []
for key, dim in ALL_EXTRACTORS:
    csv_path = WORK_DIR / f"cached_dicom_features_{key}.csv"
    meta_path = WORK_DIR / f"cached_dicom_features_{key}_metadata.json"
    meta = json.loads(meta_path.read_text(encoding="utf-8")) if meta_path.exists() else {}
    vc = meta.get("validation_checks") or {}
    summary_rows.append(
        {
            "extractor": key,
            "role": "EXTRA_CONTROL" if meta.get("is_control") else "MEDICAL_PRETRAINED",
            "status": meta.get("status", "missing"),
            "patients": meta.get("patients_processed_count", 0),
            "skipped": meta.get("patients_skipped_count", 0),
            "embed_dim": dim,
            "csv_mb": vc.get("file_size_mb", round(csv_path.stat().st_size / (1024 ** 2), 3) if csv_path.exists() else None),
            "nan_cols": vc.get("nan_columns_count"),
            "const_cols": vc.get("constant_columns_count"),
            "runtime_min": meta.get("runtime_minutes"),
            "peak_gpu_mb": meta.get("peak_gpu_memory_mb"),
            "reason_if_not_run": meta.get("not_run_reason"),
        }
    )

df_summary = pd.DataFrame(summary_rows)
print("=" * 95)
print("PULMOSIGHT ALTERNATIVE CT EXTRACTORS — FINAL CONSOLIDATED REPORT")
print("=" * 95)
print(df_summary.to_string(index=False))

print("\nOutput files ready in /kaggle/working:")
for p in sorted(WORK_DIR.glob("cached_dicom_features_*")):
    print(f"  - {p.name:58s} ({p.stat().st_size / (1024 ** 2):.3f} MB)")